<a href="https://colab.research.google.com/github/coolboy-dev/Amazon-ML-challenge-2025/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importing the dataset
import pandas as pd
df = pd.read_csv('/content/drive/My Drive/amazon/images_test/processed_train.csv')
print(f"DataFrame Headers: {list(df.columns)}")

In [ ]:
# The -q flag means 'quiet' so it doesn't print all 75,000 filenames
!unzip -q "/content/drive/My Drive/amazon/images.zip" -d "/content/images/"

In [ ]:
# Install dependencies
!pip install torch torchvision pillow pandas numpy tqdm scikit-learn

# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

# Check environment
print("="*70)
print("SETUP CHECK")
print("="*70)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU found - will use CPU (slower)")
print("="*70)


class OptimizedDINOv2Extractor:
    """
    Production-ready DINOv2 extractor
    Optimizations:
    - Batch processing (3x faster)
    - Mixed precision (2x faster)
    - Error handling
    - Progress tracking
    - Memory management
    """

    def __init__(self, model_size='small', batch_size=32):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.batch_size = batch_size

        print(f"\n🔧 Loading DINOv2-{model_size}...")

        # Load model
        model_name = f'dinov2_vit{model_size[0]}14'
        try:
            self.model = torch.hub.load('facebookresearch/dinov2', model_name, trust_repo=True)
        except Exception as e:
            print(f"Error loading from torch hub: {e}")
            print("Trying alternative method...")
            # Fallback: load from local cache
            self.model = torch.hub.load('facebookresearch/dinov2', model_name,
                                       trust_repo=True, force_reload=True)

        self.model = self.model.to(self.device)
        self.model.eval()

        # Enable mixed precision if GPU available
        self.use_amp = torch.cuda.is_available()

        # Embedding dimensions
        embed_dims = {'small': 384, 'base': 768, 'large': 1024, 'giant': 1536}
        self.embedding_dim = embed_dims[model_size]

        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        print(f"✓ Model loaded on {self.device}")
        print(f"  Embedding dim: {self.embedding_dim}")
        print(f"  Batch size: {self.batch_size}")
        print(f"  Mixed precision: {self.use_amp}")

    def load_image(self, img_path):
        """Load and transform single image"""
        try:
            img = Image.open(img_path).convert('RGB')
            return self.transform(img)
        except Exception as e:
            # Return black image if loading fails
            return torch.zeros(3, 224, 224)

    def extract_batch(self, batch_images):
        """Extract embeddings for a batch of images"""
        batch_tensor = torch.stack(batch_images).to(self.device)

        with torch.no_grad():
            if self.use_amp:
                with torch.cuda.amp.autocast():
                    embeddings = self.model(batch_tensor)
            else:
                embeddings = self.model(batch_tensor)

        return embeddings.cpu().numpy()

    def process_dataset(self, df, image_dir, save_path='dinov2_embeddings.pkl'):
        """
        Process entire dataset with optimizations
        """
        print("\n" + "="*70)
        print("STARTING EMBEDDING EXTRACTION")
        print("="*70)

        image_dir = Path(image_dir)

        # Verify image directory exists
        if not image_dir.exists():
            raise FileNotFoundError(f"Image directory not found: {image_dir}")

        # Prepare lists
        embeddings_list = []
        prices_list = []
        product_ids_list = []
        valid_indices = []
        failed_count = 0

        # Create batches
        total_samples = len(df)
        batch_images = []
        batch_data = []

        print(f"\nProcessing {total_samples} products...")
        print(f"Image directory: {image_dir}")

        start_time = time.time()
        processed = 0

        # Progress bar
        pbar = tqdm(total=total_samples, desc="Extracting embeddings")

        for idx, row in df.iterrows():
            img_path = image_dir / row['image_filename']

            # Load image
            if img_path.exists():
                img_tensor = self.load_image(str(img_path))
                batch_images.append(img_tensor)
                batch_data.append({
                    'idx': idx,
                    'price': row['price'],
                    'id': row['id']
                })
            else:
                failed_count += 1
                pbar.update(1)
                continue

            # Process batch when full
            if len(batch_images) == self.batch_size:
                try:
                    # Extract embeddings
                    batch_embeddings = self.extract_batch(batch_images)

                    # Store results
                    embeddings_list.append(batch_embeddings)
                    for data in batch_data:
                        prices_list.append(data['price'])
                        product_ids_list.append(data['id'])
                        valid_indices.append(data['idx'])

                    processed += len(batch_images)

                except Exception as e:
                    print(f"\n⚠️ Error processing batch: {e}")
                    failed_count += len(batch_images)

                # Clear batch
                batch_images = []
                batch_data = []

                # Update progress
                pbar.update(self.batch_size)

                # Memory management
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # Process remaining images
        if len(batch_images) > 0:
            try:
                batch_embeddings = self.extract_batch(batch_images)
                embeddings_list.append(batch_embeddings)
                for data in batch_data:
                    prices_list.append(data['price'])
                    product_ids_list.append(data['id'])
                    valid_indices.append(data['idx'])
                processed += len(batch_images)
            except Exception as e:
                print(f"\n⚠️ Error processing final batch: {e}")
                failed_count += len(batch_images)

            pbar.update(len(batch_images))

        pbar.close()

        # Combine all embeddings
        embeddings = np.vstack(embeddings_list) if embeddings_list else np.array([])
        prices = np.array(prices_list)
        product_ids = np.array(product_ids_list)

        elapsed = time.time() - start_time

        # Print statistics
        print("\n" + "="*70)
        print("EXTRACTION COMPLETE")
        print("="*70)
        print(f"✓ Successfully processed: {processed}/{total_samples}")
        print(f"  Failed: {failed_count}")
        print(f"  Success rate: {processed/total_samples*100:.1f}%")
        print(f"\n⏱️  Time taken: {elapsed:.2f}s ({elapsed/60:.2f} minutes)")
        print(f"  Average: {elapsed/processed*1000:.2f}ms per image")
        print(f"\n📊 Data shape:")
        print(f"  Embeddings: {embeddings.shape}")
        print(f"  Prices: {prices.shape}")
        print(f"  Size: {embeddings.nbytes / 1024 / 1024:.2f} MB")

        # Prepare data dictionary
        data = {
            'embeddings': embeddings,
            'prices': prices,
            'product_ids': product_ids,
            'valid_indices': valid_indices,
            'embedding_dim': self.embedding_dim,
            'failed_count': failed_count,
            'metadata': {
                'total_samples': total_samples,
                'processed': processed,
                'extraction_time': elapsed,
                'model': 'dinov2_small',
                'date': pd.Timestamp.now().isoformat()
            }
        }

        # Save to disk
        print(f"\n💾 Saving embeddings to {save_path}...")
        with open(save_path, 'wb') as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

        file_size = Path(save_path).stat().st_size / 1024 / 1024
        print(f"✓ Saved successfully! File size: {file_size:.2f} MB")

        # Verify saved data
        print("\n🔍 Verifying saved data...")
        with open(save_path, 'rb') as f:
            loaded_data = pickle.load(f)
        print(f"✓ Verification successful!")
        print(f"  Can load {len(loaded_data['embeddings'])} embeddings")

        print("\n" + "="*70)
        print("DAY 1 COMPLETE! 🎉")
        print("="*70)
        print(f"Next steps:")
        print(f"  1. Check the file: {save_path}")
        print(f"  2. Proceed to Day 2: Train regression head")
        print("="*70)

        return data


# ============================================
# QUICK QUALITY CHECK
# ============================================

def quick_quality_check(data_path='dinov2_embeddings.pkl'):
    """
    Verify the extracted embeddings are good quality
    """
    print("\n" + "="*70)
    print("QUALITY CHECK")
    print("="*70)

    # Load data
    with open(data_path, 'rb') as f:
        data = pickle.load(f)

    embeddings = data['embeddings']
    prices = data['prices']

    # Check 1: No NaN values
    has_nan = np.isnan(embeddings).any()
    print(f"\n1. NaN check: {'❌ Found NaN!' if has_nan else '✓ No NaN values'}")

    # Check 2: Embeddings have reasonable variance
    variances = embeddings.var(axis=0)
    avg_variance = variances.mean()
    print(f"2. Variance check: {avg_variance:.4f} {'✓ Good' if avg_variance > 0.01 else '⚠️ Low variance'}")

    # Check 3: Embeddings are normalized
    norms = np.linalg.norm(embeddings, axis=1)
    avg_norm = norms.mean()
    print(f"3. Norm check: {avg_norm:.4f} {'✓ Good' if 0.5 < avg_norm < 2.0 else '⚠️ Unusual norms'}")

    # Check 4: Price distribution
    print(f"\n4. Price statistics:")
    print(f"   Min: ${prices.min():.2f}")
    print(f"   Max: ${prices.max():.2f}")
    print(f"   Mean: ${prices.mean():.2f}")
    print(f"   Median: ${np.median(prices):.2f}")

    # Check 5: Sample similarity
    from sklearn.metrics.pairwise import cosine_similarity
    sample_indices = np.random.choice(len(embeddings), min(100, len(embeddings)), replace=False)
    sample_emb = embeddings[sample_indices]
    similarities = cosine_similarity(sample_emb)
    avg_similarity = (similarities.sum() - len(sample_indices)) / (len(sample_indices) * (len(sample_indices) - 1))
    print(f"\n5. Average cosine similarity: {avg_similarity:.4f}")
    print(f"   {'✓ Good diversity' if avg_similarity < 0.5 else '⚠️ High similarity'}")

    print("\n" + "="*70)
    print("QUALITY CHECK COMPLETE")
    print("="*70)


# ============================================
# MAIN EXECUTION
# ============================================

def main():
    """
    Run Day 1: Extract embeddings
    """

    print("\n" + "="*70)
    print("DAY 1: DINOV2 EMBEDDING EXTRACTION")
    print("Amazon ML Challenge - Price Prediction")
    print("="*70)

    # Configuration
    DRIVE = '/content/drive/My Drive/amazon/images_test/'
    TRAIN_CSV = (DRIVE + 'processed_train.csv')
    IMAGE_DIR = ('/content/images/images/')
    OUTPUT_FILE = (DRIVE + 'dinov2_embeddings.pkl')
    MODEL_SIZE = 'small'  # 'small', 'base', 'large', or 'giant'
    BATCH_SIZE = 32  # Increase if you have good GPU memory

    print(f"\n📋 Configuration:")
    print(f"  Training CSV: {TRAIN_CSV}")
    print(f"  Image directory: {IMAGE_DIR}")
    print(f"  Output file: {OUTPUT_FILE}")
    print(f"  Model: DINOv2-{MODEL_SIZE}")
    print(f"  Batch size: {BATCH_SIZE}")

    # Load dataset
    print(f"\n📂 Loading dataset...")
    try:
        df = pd.read_csv(TRAIN_CSV)
        print(f"✓ Loaded {len(df)} products")
        print(f"\nDataset info:")
        print(f"  Columns: {list(df.columns)}")
        print(f"  Sample row:")
        print(df.iloc[0])
    except FileNotFoundError:
        print(f"❌ Error: {TRAIN_CSV} not found!")
        print(f"   Make sure you have the training CSV in the current directory")
        return

    # Verify image directory
    image_path = Path(IMAGE_DIR)
    if not image_path.exists():
        print(f"\n❌ Error: Image directory '{IMAGE_DIR}' not found!")
        print(f"   Please check the path and try again")
        return

    # Count available images
    available_images = sum(1 for _, row in df.iterrows()
                          if (image_path / row['image_filename']).exists())
    print(f"\n✓ Found {available_images}/{len(df)} images ({available_images/len(df)*100:.1f}%)")

    # Initialize extractor
    extractor = OptimizedDINOv2Extractor(
        model_size=MODEL_SIZE,
        batch_size=BATCH_SIZE
    )

    # Extract embeddings
    data = extractor.process_dataset(df, IMAGE_DIR, OUTPUT_FILE)

    # Quality check
    if len(data['embeddings']) > 0:
        quick_quality_check(OUTPUT_FILE)

    print("\n✅ Day 1 Complete!")
    print(f"   Embeddings saved to: {OUTPUT_FILE}")
    print(f"   Ready for Day 2: Training regression head")


if __name__ == "__main__":
    main()

In [ ]:
import torch
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


DRIVE = '/content/drive/My Drive/amazon/images_test/'
TRAIN_CSV = '/content/train.csv'
OUTPUT_FILE = (DRIVE + 'text_embeddings_minilm.pkl')
MODEL_NAME = 'all-MiniLM-L6-v2'


def parse_and_clean_text(content):
    """
    Parses the multi-line catalog_content string into a single, clean text string.
    """
    if not isinstance(content, str):
        return ""

    lines = [line.strip() for line in content.split('\n') if line.strip()]

    text_parts = []


    for line in lines:
        if line.startswith("Item Name:"):
            text_parts.append(line.replace("Item Name:", "").strip())
        elif line.startswith("Bullet Point"):

            parts = line.split(":", 1)
            if len(parts) > 1:
                text_parts.append(parts[1].strip())
        elif line.startswith("Product Description:"):
            text_parts.append(line.replace("Product Description:", "").strip())


    return ". ".join(part for part in text_parts if part)



if __name__ == "__main__":
    print(f"📂 Loading data from '{TRAIN_CSV}'...")
    df = pd.read_csv(TRAIN_CSV)

    print("📝 Parsing and cleaning text data...")
    # Use tqdm to show progress for the apply function
    tqdm.pandas(desc="Cleaning text")
    df['cleaned_text'] = df['catalog_content'].progress_apply(parse_and_clean_text)

    print("\n✅ Text cleaning complete. Here's a sample:")
    print("-" * 50)
    print(df['cleaned_text'].iloc[1])
    print("-" * 50)

    print(f"\n🧠 Loading SentenceTransformer model: '{MODEL_NAME}'")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(MODEL_NAME, device=device)
    print(f"✓ Model loaded on {device}")

    print("\n🚀 Generating text embeddings...")
    # The .encode() method is highly optimized for this task
    text_embeddings = model.encode(
        df['cleaned_text'].tolist(),
        show_progress_bar=True,
        batch_size=128  # You can adjust this based on your GPU memory
    )

    print(f"\n📊 Text embeddings created with shape: {text_embeddings.shape}")

    # --- 4. SAVE THE RESULTS ---
    # We save embeddings and their corresponding IDs to map them later
    data_to_save = {
        'sample_ids': df['sample_id'].tolist(),
        'embeddings': text_embeddings
    }

    print(f"\n💾 Saving embeddings to '{OUTPUT_FILE}'...")
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(data_to_save, f)

    print("\n🎉 Success! Text embedding process is complete.")

In [ ]:
import pandas as pd
import numpy as np
import pickle

# --- 1. CONFIGURATION --
DRIVE = '/content/drive/My Drive/amazon/images_test/'
IMAGE_EMBEDDINGS_FILE = (DRIVE + 'dinov2_embeddings.pkl')
TEXT_EMBEDDINGS_FILE = (DRIVE + 'text_embeddings_minilm.pkl')
FINAL_OUTPUT_FILE = (DRIVE + 'final_multimodal_dataset.pkl')

# --- 2. LOAD BOTH DATASETS ---
print("📂 Loading image embeddings...")
with open(IMAGE_EMBEDDINGS_FILE, 'rb') as f:
    # Note: Your DINOv2 script saved 'product_ids', which are the sample_ids
    image_data = pickle.load(f)
df_image = pd.DataFrame({
    'sample_id': image_data['product_ids'],
    'price': image_data['prices'],
    'image_embedding': list(image_data['embeddings'])
})
print(f"✓ Found {len(df_image)} image embeddings.")

print("\n📂 Loading text embeddings...")
with open(TEXT_EMBEDDINGS_FILE, 'rb') as f:
    text_data = pickle.load(f)
df_text = pd.DataFrame({
    'sample_id': text_data['sample_ids'],
    'text_embedding': list(text_data['embeddings'])
})
print(f"✓ Found {len(df_text)} text embeddings.")


# --- 3. MERGE THE DATAFRAMES ---
print("\n🔗 Merging datasets using 'sample_id'...")
# An 'inner' merge ensures we only keep IDs that exist in BOTH files.
final_df = pd.merge(df_image, df_text, on='sample_id', how='inner')


print(f"✓ Merge complete. Final dataset has {len(final_df)} matched products.")


# --- 4. COMBINE THE EMBEDDING VECTORS ---
print("\n🧬 Concatenating image and text vectors...")
# We'll loop through the merged dataframe and stack the embeddings horizontally.
# np.hstack is perfect for this.
combined_embeddings = np.hstack((
    np.vstack(final_df['image_embedding']),
    np.vstack(final_df['text_embedding'])
))

print(f"✓ Concatenation complete.")
print(f"  Final embedding dimension: {combined_embeddings.shape[1]}")


# --- 5. SAVE THE FINAL DATASET ---
# This dictionary contains everything needed for Day 2.
final_data_to_save = {
    'sample_ids': final_df['sample_id'].tolist(),
    'embeddings': combined_embeddings,
    'prices': final_df['price'].values
}

print(f"\n💾 Saving final dataset to '{FINAL_OUTPUT_FILE}'...")
with open(FINAL_OUTPUT_FILE, 'wb') as f:
    pickle.dump(final_data_to_save, f)

print("\n🎉 All data processing is complete! You are officially ready to train your model.")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pickle
from sklearn.model_selection import train_test_split

# --- 1. CONFIGURATION & HYPERPARAMETERS ---

DRIVE = '/content/drive/My Drive/amazon/images_test/'
DATASET_FILE = (DRIVE + 'final_multimodal_dataset.pkl')
LEARNING_RATE = 0.001
BATCH_SIZE = 256
EPOCHS = 20 # Train for a few epochs; this task converges quickly
TEST_SPLIT_SIZE = 0.2 # Use 20% of data for validation

# --- 2. LOAD AND PREPARE DATA ---
print(f"📂 Loading final dataset from '{DATASET_FILE}'...")
with open(DATASET_FILE, 'rb') as f:
    data = pickle.load(f)

embeddings = data['embeddings'].astype(np.float32)
prices = data['prices'].astype(np.float32).reshape(-1, 1) # Reshape for PyTorch

print(f"✓ Loaded {len(embeddings)} total samples.")
print(f"   Embedding dimension: {embeddings.shape[1]}")

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    embeddings, prices, test_size=TEST_SPLIT_SIZE, random_state=42
)
print(f"\nSplit data into {len(X_train)} training and {len(X_val)} validation samples.")

# Create PyTorch Datasets and DataLoaders
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- 3. DEFINE THE REGRESSION HEAD MODEL ---
# Input dimension is 384 (DINOv2) + 384 (MiniLM) = 768
INPUT_DIM = embeddings.shape[1]

class RegressionHead(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Dropout helps prevent overfitting
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1) # Final output is a single number (the price)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# --- 4. TRAINING SETUP ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = RegressionHead(INPUT_DIM).to(device)
# Use L1Loss because it directly corresponds to Mean Absolute Error (MAE)
loss_function = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print(f"\n🚀 Starting training on {device}...")

# --- 5. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for batch_embeddings, batch_prices in train_loader:
        batch_embeddings, batch_prices = batch_embeddings.to(device), batch_prices.to(device)

        # Forward pass
        outputs = model(batch_embeddings)
        loss = loss_function(outputs, batch_prices)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_embeddings, batch_prices in val_loader:
            batch_embeddings, batch_prices = batch_embeddings.to(device), batch_prices.to(device)
            outputs = model(batch_embeddings)
            loss = loss_function(outputs, batch_prices)
            total_val_loss += loss.item()

    avg_val_mae = total_val_loss / len(val_loader)

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Validation MAE: ${avg_val_mae:.4f}")

print("\n🎉 Training complete!")

In [ ]:
MODEL_PATH = DRIVE + 'regression_head_model.pth'
torch.save(model.state_dict(), MODEL_PATH)
print(f"✅ Model saved at: {MODEL_PATH}")


In [ ]:
import pandas as pd
import requests
from PIL import Image
import io
from pathlib import Path
from tqdm.auto import tqdm

# --- CONFIGURATION ---
CSV_FILE = '/content/sample_test.csv'
IMAGE_DIR = '/content/test_images/' # The folder where we'll save the images

# --- MAIN SCRIPT ---
# 1. Create the directory if it doesn't exist
Path(IMAGE_DIR).mkdir(exist_ok=True)
print(f"📁 Images will be saved to '{IMAGE_DIR}'")

# 2. Load the CSV
df = pd.read_csv(CSV_FILE)
print(f"Found {len(df)} image links to download.")

# 3. Loop, download, and save
success_count = 0
fail_count = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Downloading images"):
    image_url = row['image_link']
    # Use the sample_id as the filename for easy mapping
    image_filename = f"{row['sample_id']}.jpg"
    save_path = Path(IMAGE_DIR) / image_filename

    try:
        # Send a request to the URL
        response = requests.get(image_url, timeout=10)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        # Open the image from the response content
        image = Image.open(io.BytesIO(response.content)).convert("RGB")

        # Save the image as a JPEG
        image.save(save_path, 'jpeg')
        success_count += 1

    except Exception as e:
        fail_count += 1
        # Uncomment the line below if you want to see errors as they happen
        # print(f"Failed to download {image_url}: {e}")

print("\n--- DOWNLOAD COMPLETE ---")
print(f"✅ Success: {success_count}/{len(df)}")
print(f"❌ Failed:  {fail_count}/{len(df)}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import pickle
import time
import warnings
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION ---
TEST_CSV = 'sample_test.csv'                # Your test file
IMAGE_DIR = 'test_images/'                  # Folder with your downloaded test images
MODEL_SAVE_PATH = (DRIVE + 'regression_head_model.pth')          # Your saved regression model
DINO_MODEL_SIZE = 'small'
TEXT_MODEL_NAME = 'all-MiniLM-L6-v2'
BATCH_SIZE = 64

# --- 2. PASTE IN ALL CLASS AND FUNCTION DEFINITIONS ---

# --- DINOv2 EXTRACTOR CLASS ---
class OptimizedDINOv2Extractor:
    def __init__(self, model_size='small', batch_size=32):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.batch_size = batch_size
        model_name = f'dinov2_vit{model_size[0]}14'
        self.model = torch.hub.load('facebookresearch/dinov2', model_name, trust_repo=True).to(self.device).eval()
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def load_image(self, img_path):
        try:
            img = Image.open(img_path).convert('RGB')
            return self.transform(img)
        except Exception:
            return torch.zeros(3, 224, 224)

    def extract_embeddings(self, df, image_dir):
        image_dir = Path(image_dir)
        all_embeddings = {}
        batch_images = []
        batch_ids = []

        for index, row in tqdm(df.iterrows(), total=len(df), desc="Generating Image Embeddings"):
            img_path = image_dir / f"{row['sample_id']}.jpg"
            if img_path.exists():
                batch_images.append(self.load_image(str(img_path)))
                batch_ids.append(row['sample_id'])

            if len(batch_images) == self.batch_size:
                batch_tensor = torch.stack(batch_images).to(self.device)
                with torch.no_grad(), torch.cuda.amp.autocast():
                    embeddings = self.model(batch_tensor).cpu().numpy()
                for i, sample_id in enumerate(batch_ids):
                    all_embeddings[sample_id] = embeddings[i]
                batch_images, batch_ids = [], []

        if len(batch_images) > 0:
            batch_tensor = torch.stack(batch_images).to(self.device)
            with torch.no_grad(), torch.cuda.amp.autocast():
                embeddings = self.model(batch_tensor).cpu().numpy()
            for i, sample_id in enumerate(batch_ids):
                all_embeddings[sample_id] = embeddings[i]

        return all_embeddings

# --- TEXT PARSING FUNCTION ---
def parse_and_clean_text(content):
    if not isinstance(content, str): return ""
    lines = [line.strip() for line in content.split('\n') if line.strip()]
    text_parts = []
    for line in lines:
        if line.startswith("Item Name:"): text_parts.append(line.replace("Item Name:", "").strip())
        elif line.startswith("Bullet Point"):
            parts = line.split(":", 1)
            if len(parts) > 1: text_parts.append(parts[1].strip())
        elif line.startswith("Product Description:"): text_parts.append(line.replace("Product Description:", "").strip())
    return ". ".join(part for part in text_parts if part)

# --- REGRESSION HEAD MODEL CLASS ---
INPUT_DIM = 768

class RegressionHead(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x



# --- 3. MAIN EXECUTION SCRIPT ---
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"--- Starting Test Pipeline on {device} ---")

    # 1. Load the test CSV
    df_test = pd.read_csv(TEST_CSV)

    # 2. Generate Image Embeddings
    dino_extractor = OptimizedDINOv2Extractor(model_size=DINO_MODEL_SIZE, batch_size=BATCH_SIZE)
    image_embeddings_dict = dino_extractor.extract_embeddings(df_test, IMAGE_DIR)
    df_image = pd.DataFrame(list(image_embeddings_dict.items()), columns=['sample_id', 'image_embedding'])

    # 3. Generate Text Embeddings
    tqdm.pandas(desc="Cleaning text")
    df_test['cleaned_text'] = df_test['catalog_content'].progress_apply(parse_and_clean_text)

    text_model = SentenceTransformer(TEXT_MODEL_NAME, device=device)
    text_embeddings = text_model.encode(
        df_test['cleaned_text'].tolist(),
        show_progress_bar=True,
        batch_size=BATCH_SIZE
    )
    df_text = pd.DataFrame({
        'sample_id': df_test['sample_id'],
        'text_embedding': list(text_embeddings)
    })

    # 4. Merge Embeddings and Prepare for Model
    print("\n🔗 Merging image and text embeddings...")
    df_merged = pd.merge(df_image, df_text, on='sample_id', how='inner')

    combined_embeddings = np.hstack((
        np.vstack(df_merged['image_embedding']),
        np.vstack(df_merged['text_embedding'])
    )).astype(np.float32)

    # 5. Load Trained Model and Make Predictions
    print("🧠 Loading trained regression model...")
    model = RegressionHead(INPUT_DIM).to(device)
    model.load_state_dict(torch.load(MODEL_SAVE_PATH))
    model.eval()

    print("🚀 Making final price predictions...")
    test_tensor = torch.from_numpy(combined_embeddings).to(device)

    predictions = {}
    with torch.no_grad():
        log_predictions = model(test_tensor)
        # Convert predictions back from log scale to actual dollars!
        dollar_predictions = torch.expm1(log_predictions).cpu().numpy().flatten()

        for i, sample_id in enumerate(df_merged['sample_id']):
            predictions[sample_id] = dollar_predictions[i]

    # 6. Save Predictions back to the original CSV
    print(f"💾 Saving predictions back to '{TEST_CSV}'...")
    df_test['price'] = df_test['sample_id'].map(predictions)

    # Fill any failed predictions with a default value like the mean, or 0
    df_test['price'].fillna(0, inplace=True)

    df_test.to_csv(TEST_CSV, index=False)

    print("\n🎉 --- PIPELINE COMPLETE --- 🎉")
    print(f"Successfully updated '{TEST_CSV}' with a new 'price' column.")